In [2]:
import torch
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import math as py_math
from kan import KAN  # Importing the Kolmogorov-Arnold Network


DATA_PATH = "Data/burgers_data.npy"

In [3]:
def prepare_data(file_path=DATA_PATH, t_split=0.5, spatial_mask_ratio=0.8):
    """
    Loads data and performs strict spatio-temporal splitting to prevent leakage.
    Maintains the exact same data epistemology as our prior MLP benchmarks.
    """
    data = np.load(file_path, allow_pickle=True).item()
    x_all = data['x']
    t_all = data['t']
    u_all = data['u']
    
    # Temporal Split (Forecasting Validation)
    train_time_mask = (t_all <= t_split).flatten()
    test_time_mask = (t_all > t_split).flatten()
    
    x_train_temp = x_all[train_time_mask]
    t_train_temp = t_all[train_time_mask]
    u_train_temp = u_all[train_time_mask]
    
    x_test = x_all[test_time_mask]
    t_test = t_all[test_time_mask]
    u_test = u_all[test_time_mask]

    # Spatial Masking (Interpolation Validation)
    num_train_points = len(x_train_temp)
    indices = np.arange(num_train_points)
    np.random.shuffle(indices)
    
    keep_len = int(num_train_points * spatial_mask_ratio)
    train_indices = indices[:keep_len]
    
    x_train = x_train_temp[train_indices]
    t_train = t_train_temp[train_indices]
    u_train = u_train_temp[train_indices]
    
    # Meshless Collocation Points (For PDE Loss)
    n_colloc = 10000
    x_colloc = np.random.uniform(-1, 1, (n_colloc, 1))
    t_colloc = np.random.uniform(0, 1, (n_colloc, 1))

    to_tensor = lambda arr: torch.tensor(arr, dtype=torch.float32)
    
    return {
        'train': (to_tensor(x_train), to_tensor(t_train), to_tensor(u_train)),
        'test': (to_tensor(x_test), to_tensor(t_test), to_tensor(u_test)),
        'colloc': (to_tensor(x_colloc), to_tensor(t_colloc))
    }

In [4]:
def compute_physics_loss(model, x, t, nu):
    """
    Evaluates the Burgers' equation residual exactly as before.
    The continuous B-splines of the KAN support create_graph=True natively.
    """
    x.requires_grad_(True)
    t.requires_grad_(True)
    
    # KAN models require concatenated inputs across the feature dimension
    inputs = torch.cat([x, t], dim=1)
    u = model(inputs)
    
    u_x = torch.autograd.grad(u, x, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_t = torch.autograd.grad(u, t, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, x, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]
    
    residual = u_t + u * u_x - nu * u_xx
    return torch.mean(residual**2)

In [ ]:
def train_pi_kan():
    device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.mps.is_available() else "cpu"))
    print(f"Executing Physics-Informed Optimization on: {device}")
    
    nu_true = 0.01 / py_math.pi
    data_splits = prepare_data(t_split=0.5, spatial_mask_ratio=0.8)
    
    x_train, t_train, u_train = [t.to(device) for t in data_splits['train']]
    x_colloc, t_colloc = [t.to(device) for t in data_splits['colloc']]
    
    # Initialize the KAN
    # width=[2 inputs, 20 hidden, 20 hidden, 1 output]
    # grid=5 provides base spatial flexibility, k=3 ensures continuous second derivatives
    model_kan = KAN(width=[2, 20, 20, 1], grid=5, k=3).to(device)
    
    # We use Adam for the initial manifold descent
    optimizer = optim.Adam(model_kan.parameters(), lr=1e-3)
    
    epochs = 3000
    history = {'data_loss': [], 'pde_loss': []}

    model_kan.train()
    
    for epoch in range(epochs):
        optimizer.zero_grad()
        
        # Supervised Loss (Data Fidelity)
        train_inputs = torch.cat([x_train, t_train], dim=1)
        u_pred = model_kan(train_inputs)
        loss_data = torch.nn.functional.mse_loss(u_pred, u_train)
        
        # Unsupervised Loss (Physical Residuals)
        loss_pde = compute_physics_loss(model_kan, x_colloc, t_colloc, nu=nu_true)
        
        # Composite Optimization
        loss_total = loss_data + loss_pde
        loss_total.backward()
        optimizer.step()
        
        # Logging
        history['data_loss'].append(loss_data.item())
        history['pde_loss'].append(loss_pde.item())
        
        if epoch % 500 == 0:
            print(f"Epoch {epoch:04d} | Data Loss: {loss_data.item():.6e} | PDE Loss: {loss_pde.item():.6e}")

    # Optional: pykan provides a method to update grids if the data domain shifts, 
    # but for static domains like [-1, 1], the default initialization is generally sufficient.

    print("PI-KAN Training Complete.")
    return model_kan, data_splits, device, history

# Execute the KAN pipeline
trained_kan, splits, comp_device, training_history = train_pi_kan()

Executing Physics-Informed Optimization on: mps
checkpoint directory created: ./model
saving model version 0.0
Epoch 0000 | Data Loss: 4.701062e-01 | PDE Loss: 2.941910e-04


KeyboardInterrupt: 